# Earnings Yield Signal — Validation Notebook

**Formula:** E/P = TTM EPS / current price  
**Why E/P not P/E:** Handles negative EPS correctly (ranks as poor value, not "ultra-cheap").  
**No separate DB table** — values consumed by scoring engine directly.

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
from signals.earnings_yield import compute_earnings_yield
from db import read_sql

df = compute_earnings_yield()
ey = df["earnings_yield"].dropna()
print(f"Earnings Yield: {len(df)} stocks, {len(ey)} computed")
print(f"  Mean E/P: {ey.mean():.4f} ({1/ey[ey>0].mean():.1f}x implied P/E)")
print(f"  Median E/P: {ey.median():.4f}")
print(f"  Negative EPS: {(ey < 0).sum()} stocks")
print(f"\nPercentiles:")
for p in [5, 25, 50, 75, 95]:
    print(f"  {p}th: {ey.quantile(p/100):.4f}")

# Per-tier breakdown
tiers = read_sql("SELECT sid, cap_tier FROM stocks")
dfm = df.merge(tiers, on="sid")
for tier in ["LARGE", "MID", "SMALL"]:
    t = dfm[dfm["cap_tier"] == tier]["earnings_yield"].dropna()
    if len(t) > 0:
        implied_pe = 1 / t[t > 0].mean() if (t > 0).any() else float("nan")
        print(f"  {tier}: {len(t)} stocks, median E/P={t.median():.4f} ({implied_pe:.1f}x P/E)")